In [0]:
%run ./00_config

In [0]:
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, upper, lit
from pyspark.sql.types import (
    DateType,
    TimestampType,
    StringType,
    DoubleType,
    IntegerType,
    LongType,
    FloatType,
    DecimalType,
)

# ============================================================================
# PRODUCTION BILLING SILVER TRANSFORMATION
# Bronze  : billing.sales_data.billing_bronze
# Silver  : billing.sales_data.billing_silver
# Strategy: deterministic transformation + atomic date-level replacement
# ============================================================================

PIPELINE_RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc)

BRONZE_TABLE = f"{catalog_name}.{schema_name}.billing_bronze"
SILVER_TABLE = f"{catalog_name}.{schema_name}.billing_silver"

print("=" * 80)
print("PRODUCTION BILLING SILVER TRANSFORMATION")
print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
print(f"Target date     : {TARGET_DATE}")
print(f"Source table    : {BRONZE_TABLE}")
print(f"Target table    : {SILVER_TABLE}")
print(f"Started at      : {STARTED_AT.isoformat()}")
print("=" * 80)


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def table_exists(table_name: str) -> bool:
    """Return True only when the target table actually exists."""
    try:
        return bool(spark.catalog.tableExists(table_name))
    except Exception:
        # Catalog APIs can behave differently with three-part identifiers.
        # Fall back to a metadata query without interpreting arbitrary
        # AnalysisException as 'table does not exist'.
        try:
            spark.sql(f"DESCRIBE TABLE {table_name}")
            return True
        except Exception:
            return False


def normalize_source_date_for_existing_schema(df, target_date, target_type):
    """Add source_date using the existing Silver column type."""
    if isinstance(target_type, DateType):
        return df.withColumn("source_date", F.to_date(F.lit(target_date)))

    if isinstance(target_type, TimestampType):
        return df.withColumn(
            "source_date",
            F.to_timestamp(F.lit(f"{target_date} 00:00:00"), "yyyy-MM-dd HH:mm:ss"),
        )

    # Existing Bronze/Silver deployments commonly have source_date as STRING.
    if isinstance(target_type, StringType):
        return df.withColumn("source_date", F.lit(str(target_date)))

    raise RuntimeError(
        f"Unsupported existing Silver source_date type: {target_type}. "
        "Expected STRING, DATE, or TIMESTAMP."
    )


def source_date_replace_predicate(target_date, source_date_type):
    """Build a replaceWhere predicate compatible with the existing column type."""
    escaped_date = str(target_date).replace("'", "''")

    if isinstance(source_date_type, DateType):
        return f"source_date = DATE '{escaped_date}'"

    if isinstance(source_date_type, TimestampType):
        return f"source_date = TIMESTAMP '{escaped_date} 00:00:00'"

    if isinstance(source_date_type, StringType):
        return f"source_date = '{escaped_date}'"

    raise RuntimeError(
        f"Unsupported Silver source_date type for replaceWhere: {source_date_type}"
    )


def parse_date_safe(column_name):
    """Parse supported date formats without ANSI-mode parser failures."""
    value = F.trim(F.col(column_name).cast("string"))

    return (
        F.when(
            value.rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.to_date(value, "yyyy-MM-dd"),
        )
        .when(
            value.rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
            F.to_date(value, "yyyy/M/d"),
        )
        .when(
            value.rlike(r"^\d{1,2}/\d{1,2}/\d{4}$"),
            F.to_date(value, "M/d/yyyy"),
        )
        .when(
            value.rlike(r"^\d{1,2}/\d{1,2}/\d{2}$"),
            F.to_date(value, "M/d/yy"),
        )
        .otherwise(F.lit(None).cast("date"))
    )


def parse_timestamp_safe(column_name):
    """Parse a timestamp without allowing malformed values to abort the job."""
    value = F.trim(F.col(column_name).cast("string"))

    # Databricks supports try_to_timestamp in current runtimes. Using expr
    # avoids Python-side parsing and keeps this operation distributed.
    return F.expr(f"try_to_timestamp(`{column_name}`)")


def cast_numeric_safe(column_name, target_type):
    """ANSI-safe numeric cast. Invalid numeric values become NULL."""
    return F.expr(
        f"try_cast(`{column_name}` AS {target_type})"
    )


def validate_required_columns(df, required_columns, stage):
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"Missing required columns at {stage}: {missing}"
        )


def validate_no_unexpected_zero(df, stage):
    count = df.count()
    if count == 0:
        raise RuntimeError(f"{stage} produced 0 rows — aborting before write")
    return count


# ----------------------------------------------------------------------------
# 1. Read Bronze snapshot
# ----------------------------------------------------------------------------
if not table_exists(BRONZE_TABLE):
    raise RuntimeError(f"Bronze table does not exist: {BRONZE_TABLE}")

bronze_table_df = spark.table(BRONZE_TABLE)

if "source_date" not in bronze_table_df.columns:
    raise RuntimeError(
        f"Bronze table {BRONZE_TABLE} does not contain required source_date column"
    )

bronze_source_date_type = bronze_table_df.schema["source_date"].dataType

bronze_df = bronze_table_df.filter(
    col("source_date").cast("string") == str(TARGET_DATE)
)

bronze_count = bronze_df.count()
print(f"📥 Bronze rows for {TARGET_DATE}: {bronze_count:,}")

if bronze_count == 0:
    raise RuntimeError(
        f"❌ No Bronze data found for {TARGET_DATE} — did the Bronze ingestion run?"
    )


# ----------------------------------------------------------------------------
# 2. Validate source columns before transformations
# ----------------------------------------------------------------------------
required_columns = [
    "invoice",
    "productId",
    "orderStatus",
    "orderDate",
    "time",
    "quantity",
    "productMrp",
    "sellingPrice",
    "discountAmount",
    "totalProductPrice",
    "costPrice",
    "deliveryFee",
    "GSTAmount",
    "CGSTAmount",
    "SGSTAmount",
    "cessAmount",
    "orderAmountTax",
    "orderAmountNet",
    "cashAmount",
    "cardAmount",
    "upiAmount",
    "creditAmount",
    "orderType",
    "paymentMethod",
    "brandName",
    "categoryName",
    "storeName",
    "productName",
]

validate_required_columns(bronze_df, required_columns, "Bronze input")


# ----------------------------------------------------------------------------
# 3. Deterministic Silver transformation
# ----------------------------------------------------------------------------
# Keep the user's existing business rule exactly:
# one row per (invoice, productId) after duplicate removal and required-key
# filtering.
silver_df = (
    bronze_df
    .dropDuplicates(["invoice", "productId"])
    .filter(col("orderStatus").isNotNull())
    .filter(col("invoice").isNotNull())
    .filter(col("productId").isNotNull())
    .withColumn("orderDate", parse_date_safe("orderDate"))
    .withColumn("time", parse_timestamp_safe("time"))
    .withColumn("quantity", cast_numeric_safe("quantity", "INT"))
    .withColumn("productMrp", cast_numeric_safe("productMrp", "DOUBLE"))
    .withColumn("sellingPrice", cast_numeric_safe("sellingPrice", "DOUBLE"))
    .withColumn("discountAmount", cast_numeric_safe("discountAmount", "DOUBLE"))
    .withColumn("totalProductPrice", cast_numeric_safe("totalProductPrice", "DOUBLE"))
    .withColumn("costPrice", cast_numeric_safe("costPrice", "DOUBLE"))
    .withColumn("deliveryFee", cast_numeric_safe("deliveryFee", "DOUBLE"))
    .withColumn("GSTAmount", cast_numeric_safe("GSTAmount", "DOUBLE"))
    .withColumn("CGSTAmount", cast_numeric_safe("CGSTAmount", "DOUBLE"))
    .withColumn("SGSTAmount", cast_numeric_safe("SGSTAmount", "DOUBLE"))
    .withColumn("cessAmount", cast_numeric_safe("cessAmount", "DOUBLE"))
    .withColumn("orderAmountTax", cast_numeric_safe("orderAmountTax", "DOUBLE"))
    .withColumn("orderAmountNet", cast_numeric_safe("orderAmountNet", "DOUBLE"))
    .withColumn("cashAmount", cast_numeric_safe("cashAmount", "DOUBLE"))
    .withColumn("cardAmount", cast_numeric_safe("cardAmount", "DOUBLE"))
    .withColumn("upiAmount", cast_numeric_safe("upiAmount", "DOUBLE"))
    .withColumn("creditAmount", cast_numeric_safe("creditAmount", "DOUBLE"))
    .withColumn("orderStatus", upper(trim(col("orderStatus"))))
    .withColumn("orderType", upper(trim(col("orderType"))))
    .withColumn("paymentMethod", upper(trim(col("paymentMethod"))))
    .withColumn("brandName", trim(col("brandName")))
    .withColumn("categoryName", trim(col("categoryName")))
    .withColumn("storeName", trim(col("storeName")))
    .withColumn("productName", trim(col("productName")))
)

# Do not blindly assume the existing Silver source_date type. The existing
# Bronze table in this deployment uses STRING, and the Silver table may also
# already exist with STRING. The helper preserves that established contract.

silver_count_before_write = validate_no_unexpected_zero(
    silver_df,
    "Silver transformation",
)

print(
    f"🧹 Silver transformation complete: {silver_count_before_write:,} rows "
    f"before source_date assignment"
)


# ----------------------------------------------------------------------------
# 4. Create or conform to the Silver table schema
# ----------------------------------------------------------------------------
if table_exists(SILVER_TABLE):
    existing_silver = spark.table(SILVER_TABLE)
    existing_schema = existing_silver.schema
    existing_columns = [f.name for f in existing_schema.fields]

    if "source_date" not in existing_columns:
        raise RuntimeError(
            f"Existing Silver table {SILVER_TABLE} does not contain source_date"
        )

    silver_source_date_type = existing_schema["source_date"].dataType

    # Add source_date using the exact type already present in Silver.
    silver_df = normalize_source_date_for_existing_schema(
        silver_df,
        TARGET_DATE,
        silver_source_date_type,
    )

    # Existing Silver tables are contracts. Reorder/select exactly to the
    # existing schema and cast compatible columns where necessary.
    transformed_columns = set(silver_df.columns)
    missing_existing_columns = [
        c for c in existing_columns
        if c not in transformed_columns
        and c not in ("ingested_at",)
    ]

    if missing_existing_columns:
        raise RuntimeError(
            "Existing Silver schema contains columns that the transformation "
            f"does not produce: {missing_existing_columns}"
        )

    # Preserve audit timestamp if the table has one; otherwise create it only
    # when the schema explicitly requires it.
    if "ingested_at" in existing_columns:
        silver_df = silver_df.withColumn("ingested_at", F.current_timestamp())

    # Conform to the existing column order and data types. source_date has
    # already been created with its exact existing type.
    select_exprs = []
    for field in existing_schema.fields:
        name = field.name
        if name not in silver_df.columns:
            if field.nullable:
                select_exprs.append(F.lit(None).cast(field.dataType).alias(name))
            else:
                raise RuntimeError(
                    f"Required non-nullable Silver column '{name}' is missing"
                )
        else:
            if name == "source_date" or name == "ingested_at":
                select_exprs.append(col(name).cast(field.dataType).alias(name))
            else:
                select_exprs.append(col(name).cast(field.dataType).alias(name))

    silver_df = silver_df.select(*select_exprs)

    replace_predicate = source_date_replace_predicate(
        TARGET_DATE,
        silver_source_date_type,
    )

    print("=" * 80)
    print("ATOMIC SILVER REPLACEMENT")
    print(f"Existing source_date type: {silver_source_date_type.simpleString()}")
    print(f"replaceWhere: {replace_predicate}")
    print("=" * 80)

    # Atomic Delta transaction. This replaces only the requested date and
    # avoids the unsafe DELETE + APPEND sequence.
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", replace_predicate)
        .saveAsTable(SILVER_TABLE)
    )

    print("✅ Atomic Delta Silver replacement committed")

else:
    # First creation. Use STRING source_date to match the current Bronze
    # contract and avoid introducing an unexpected schema mismatch between
    # Bronze and Silver in this deployment.
    silver_df = silver_df.withColumn("source_date", F.lit(str(TARGET_DATE)))
    silver_df = silver_df.withColumn("ingested_at", F.current_timestamp())

    print("ℹ️ Silver table does not exist — creating initial Delta table")
    print("   source_date type: STRING")

    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("source_date")
        .saveAsTable(SILVER_TABLE)
    )

    print("✅ Initial Silver table created")


# ----------------------------------------------------------------------------
# 5. Post-write validation
# ----------------------------------------------------------------------------
final_df = spark.table(SILVER_TABLE)

if "source_date" not in final_df.columns:
    raise RuntimeError(
        f"Silver table {SILVER_TABLE} is missing source_date after write"
    )

actual_count = final_df.filter(
    col("source_date").cast("string") == str(TARGET_DATE)
).count()

print("=" * 80)
print("SILVER POST-WRITE VALIDATION")
print(f"Expected transformed rows : {silver_count_before_write:,}")
print(f"Actual Silver rows        : {actual_count:,}")
print("=" * 80)

if actual_count != silver_count_before_write:
    raise RuntimeError(
        "Silver validation failed: expected "
        f"{silver_count_before_write:,} rows for {TARGET_DATE}, "
        f"but found {actual_count:,}."
    )

if actual_count == 0:
    raise RuntimeError("Silver write completed but 0 rows were found — aborting")

completed_at = datetime.now(timezone.utc)

print("=" * 80)
print("✅ BILLING SILVER TRANSFORMATION COMPLETED SUCCESSFULLY")
print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
print(f"Target date     : {TARGET_DATE}")
print(f"Rows written    : {actual_count:,}")
print(f"Completed at    : {completed_at.isoformat()}")
print("=" * 80)


PRODUCTION BILLING SILVER TRANSFORMATION
Pipeline run ID : 593f638f-02d4-4c01-92bf-1e8c9520a6f0
Target date     : 2026-10-03
Source table    : billing.sales_data.billing_bronze
Target table    : billing.sales_data.billing_silver
Started at      : 2026-10-04T10:15:48.895100+00:00
📥 Bronze rows for 2026-10-03: 52,995
🧹 Silver transformation complete: 52,995 rows before source_date assignment
ATOMIC SILVER REPLACEMENT
Existing source_date type: string
replaceWhere: source_date = '2026-10-03'
✅ Atomic Delta Silver replacement committed
SILVER POST-WRITE VALIDATION
Expected transformed rows : 52,995
Actual Silver rows        : 52,995
✅ BILLING SILVER TRANSFORMATION COMPLETED SUCCESSFULLY
Pipeline run ID : 593f638f-02d4-4c01-92bf-1e8c9520a6f0
Target date     : 2026-10-03
Rows written    : 52,995
Completed at    : 2026-10-04T10:16:03.616759+00:00


✅ Config loaded : billing.sales_data
📅 Target date   : 2026-10-03
